In [1]:
import os
# Prevent OpenMP library conflict crash
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import easyocr
from difflib import SequenceMatcher

In [2]:
VIDEO_PATH = r"test\Traffic Control CCTV.mp4"
OUTPUT_DIR = "results"
VEHICLE_MODEL_PATH = "yolov8n.pt"
PLATE_MODEL_PATH = "train2/weights/best.pt"

In [3]:
# Search Targets
TARGET_PLATE = "MW5I VSU"
TARGET_HEX = "#2a384e"  # Target vehicle color hex code
# Hardware 
DEVICE = "cuda"  # Change to "cpu" if running without NVIDIA GPU
USE_GPU_OCR = True
# Confidence & Thresholds
VEHICLE_CONF = 0.30
PLATE_CONF = 0.30
MIN_SAVE_SCORE = 0.30   # Minimum OCR similarity score to save the result
LOW_TH = 30             # Color matching strictness (lower = stricter)
# Performance Settings
PLATE_INTERVAL = 5      # Run plate detection only every N frames

In [4]:
def hex_to_hsv(hex_color):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    rgb_pixel = np.uint8([[[b, g, r]]]) 
    return cv2.cvtColor(rgb_pixel, cv2.COLOR_BGR2HSV)[0][0]
def get_color_info(image):
    if image is None or image.size == 0:
        return (0, 0, 0), "#000000"
    h, w = image.shape[:2]
    crop = image[h//4 : 3*h//4, w//4 : 3*w//4]
    if crop.size == 0: crop = image 
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB)
    l, a, b_chan = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    clahe_bgr = cv2.cvtColor(cv2.merge((cl, a, b_chan)), cv2.COLOR_LAB2BGR)
    small = cv2.resize(clahe_bgr, (50, 50))
    pixels = np.float32(small.reshape(-1, 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, labels, centers = cv2.kmeans(pixels, 3, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    dominant_bgr = centers[np.argmax(np.bincount(labels.flatten()))]
    dominant_hsv = cv2.cvtColor(np.uint8([[dominant_bgr]]), cv2.COLOR_BGR2HSV)[0][0]
    b_val, g_val, r_val = [int(c) for c in dominant_bgr]
    return dominant_hsv, f"#{r_val:02X}{g_val:02X}{b_val:02X}"
def hsv_distance(c1, c2):
    h1, s1 = int(c1[0]), int(c1[1])
    h2, s2 = int(c2[0]), int(c2[1])
    dh = min(abs(h1 - h2), 180 - abs(h1 - h2))
    ds = abs(s1 - s2) * 0.1  
    return np.sqrt(dh**2 + ds**2)
def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()
def clean_plate_text(text):
    text = text.upper().strip().replace(" ", "").replace("\n", "").replace("\r", "").replace("-", "")
    return "".join(c for c in text if c.isalnum())
print("Initializing models and tracker...")
vehicle_model = YOLO(VEHICLE_MODEL_PATH)
plate_model = YOLO(PLATE_MODEL_PATH)
reader = easyocr.Reader(['en'], gpu=USE_GPU_OCR)
tracker = DeepSort(max_age=30, embedder="mobilenet", embedder_gpu=(DEVICE == "cuda"))
os.makedirs(OUTPUT_DIR, exist_ok=True)
TARGET_HSV = hex_to_hsv(TARGET_HEX)
track_memory = {}
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video:\n{VIDEO_PATH}")
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_idx = 0
print(f"Starting video processing ({total_frames} frames)...")
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        display_frame = frame.copy()
        results = vehicle_model(frame, conf=VEHICLE_CONF, device=DEVICE, verbose=False)
        detections = []
        for box in results[0].boxes:
            cls, conf = int(box.cls), float(box.conf)
            label = vehicle_model.names[cls]
            if label not in ["car", "truck", "bus", "motorbike"]: continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            if x2 > x1 and y2 > y1:
                detections.append(([x1, y1, x2 - x1, y2 - y1], conf, label))
        tracks = tracker.update_tracks(detections, frame=frame)
        for track in tracks:
            if not track.is_confirmed(): continue
            track_id = track.track_id
            l, t, r, b = map(int, track.to_ltrb())
            x1, y1 = max(0, l), max(0, t)
            x2, y2 = min(frame.shape[1], r), min(frame.shape[0], b)
            if x2 <= x1 or y2 <= y1: continue
            vehicle_crop = frame[y1:y2, x1:x2]
            if vehicle_crop.size == 0: continue
            detected_hsv, detected_hex = get_color_info(vehicle_crop)
            dist = hsv_distance(detected_hsv, TARGET_HSV)
            if dist > LOW_TH: continue
            if frame_idx % PLATE_INTERVAL != 0: continue
            plate_results = plate_model(vehicle_crop, conf=PLATE_CONF, device=DEVICE, verbose=False)
            best_score, best_text, best_plate_box = 0.0, "Not detected", None
            for pbox in plate_results[0].boxes:
                px1, py1, px2, py2 = map(int, pbox.xyxy[0])
                gx1, gy1 = max(0, x1 + px1), max(0, y1 + py1)
                gx2, gy2 = min(frame.shape[1], x1 + px2), min(frame.shape[0], y1 + py2)
                if gx2 <= gx1 or gy2 <= gy1: continue
                plate_crop = frame[gy1:gy2, gx1:gx2]
                if plate_crop.size == 0: continue
                ocr_output = reader.readtext(plate_crop, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", detail=0)
                clean_text = clean_plate_text("".join(ocr_output))
                score = similarity(clean_text, TARGET_PLATE)
                if score > best_score:
                    best_score, best_text, best_plate_box = score, clean_text, (gx1, gy1, gx2, gy2)
            if best_score >= MIN_SAVE_SCORE:
                if track_id not in track_memory or best_score > track_memory[track_id]["best_score"]:
                    cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
                    if best_plate_box:
                        cv2.rectangle(display_frame, (best_plate_box[0], best_plate_box[1]), 
                                      (best_plate_box[2], best_plate_box[3]), (255, 0, 0), 2)
                    cv2.putText(display_frame, f"ID:{track_id}", (x1, max(20, y1 - 10)), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                    cv2.putText(display_frame, f"{best_text} ({best_score:.2f})", 
                                (x1, min(frame.shape[0] - 10, y2 + 25)), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    track_memory[track_id] = {
                        "best_score": best_score,
                        "plate": best_text,
                        "frame": cv2.resize(display_frame, (854, 480)),
                        "time": frame_idx / fps}
        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx}/{total_frames} frames | Tracks stored: {len(track_memory)}")
finally:
    cap.release()
    cv2.destroyAllWindows()

Initializing models and tracker...


C:\Users\Saijoshith\anaconda3\Lib\site-packages\deep_sort_realtime\embedder\embedder_pytorch.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(

Starting video processing (1800 frames)...
Processed 30/1800 frames | Tracks stored: 1
Processed 60/1800 frames | Tracks stored: 1
Processed 90/1800 frames | Tracks stored: 1
Processed 120/1800 frames | Tracks stored: 1
Processed 150/1800 frames | Tracks stored: 2
Processed 180/1800 frames | Tracks stored: 2
Processed 210/1800 frames | Tracks stored: 2
Processed 240/1800 frames | Tracks stored: 2
Processed 270/1800 frames | Tracks stored: 2
Processed 300/1800 frames | Tracks stored: 2
Processed 330/1800 frames | Tracks stored: 2
Processed 360/1800 frames | Tracks stored: 2
Processed 390/1800 frames | Tracks stored: 5
Processed 420/1800 frames | Tracks stored: 5
Processed 450/1800 frames | Tracks stored: 5
Processed 480/1800 frames | Tracks stored: 5
Processed 510/1800 frames | Tracks stored: 5
Processed 540/1800 frames | Tracks stored: 5
Processed 570/1800 frames | Tracks stored: 5
Processed 600/1800 frames | Tracks stored: 5
Processed 630/1800 frames | Tracks stored: 5
Processed 660/1

In [5]:
sorted_tracks = sorted(track_memory.items(), key=lambda x: x[1]["best_score"], reverse=True)
print("\n" + "=" * 60)
print("FINAL RANKED RESULTS")
print("=" * 60)
if not sorted_tracks:
    print("No matching vehicles/plates were found.")
else:
    for rank, (track_id, data) in enumerate(sorted_tracks, start=1):
        filename = os.path.join(OUTPUT_DIR, f"rank_{rank}_vehicle_{track_id}.jpg")
        cv2.imwrite(filename, data["frame"])
        print(f"Rank {rank} | ID:{track_id} | Plate:{data['plate']:<10} | Score:{data['best_score']:.2f} | Time:{data['time']:.2f}s")
print(f"\nSaved ranked images in: {os.path.abspath(OUTPUT_DIR)}")


FINAL RANKED RESULTS
Rank 1 | ID:5 | Plate:MEIVSU     | Score:0.71 | Time:2.33s
Rank 2 | ID:650 | Plate:M05V       | Score:0.50 | Time:45.50s
Rank 3 | ID:279 | Plate:WQILS      | Score:0.46 | Time:27.67s
Rank 4 | ID:341 | Plate:WISZZC     | Score:0.43 | Time:24.33s
Rank 5 | ID:326 | Plate:EYOMWS     | Score:0.43 | Time:30.83s
Rank 6 | ID:304 | Plate:LMISZZC    | Score:0.40 | Time:25.33s
Rank 7 | ID:8 | Plate:KISZ       | Score:0.33 | Time:5.50s
Rank 8 | ID:85 | Plate:HMI4       | Score:0.33 | Time:12.33s
Rank 9 | ID:97 | Plate:WGIG       | Score:0.33 | Time:12.50s
Rank 10 | ID:596 | Plate:CESU       | Score:0.33 | Time:53.33s
Rank 11 | ID:789 | Plate:GIBS       | Score:0.33 | Time:58.33s
Rank 12 | ID:149 | Plate:WEKGI      | Score:0.31 | Time:13.00s

Saved ranked images in: C:\Users\Saijoshith\adv_ml\results
